# Module 6: Inspectable Neo4j Memory

This notebook stores one hotel preference in Neo4j. It recalls the preference for the same actor in a new session, returns no preference for a second actor, and traces the record to its source message and `Hotel` node.

Recall scope and application authorization use two controls:

- Multi-tenant mode requires a user identifier for every memory write. The application must still authenticate each actor and authorize access to session IDs.
- In version 0.5.0, the library's semantic searches cover the entire store. This notebook scopes recall by starting each query at the selected `User`.

**Prerequisites:** The hotel graph must contain exactly one `Hotel` named `AnyCompany Cairo Nile View`. Configure Neo4j credentials for the correct database and AWS credentials that can invoke Titan Text Embeddings V2 in `AWS_REGION`. This module uses the same Neo4j instance and credentials as the earlier modules. The repository `.env` described in the top-level README provides those settings. `load_config` reads this folder's `.env` first, followed by the repository `.env`. Live cells skip when credentials are unavailable.

In [ ]:
# At an AWS event: dependencies are pre-installed. Run this cell as-is.
# Self-paced: uncomment the line below first.
# !pip install "neo4j-agent-memory[bedrock]==0.5.0" boto3 python-dotenv

print("Environment ready")

## 1. Configure one isolated workshop run

Every actor and session identifier includes a short run ID, which separates each run from earlier transcripts. All identifiers also use the `demo08-` prefix. `cleanup_memory.py` uses this shared namespace to remove every workshop run. Each live cell opens and closes its own memory client, so a failure cannot leave a connection open for the next cell.

In [ ]:
import os
import sys
from pathlib import Path


def locate_notebooks_root():
    override = os.environ.get("WORKSHOP_NOTEBOOKS_DIR")
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "workshop").is_dir():
            return candidate
        raise RuntimeError(
            "WORKSHOP_NOTEBOOKS_DIR must contain the workshop package"
        )

    start = Path.cwd().resolve()
    for candidate in (start, start / "notebooks", start.parent):
        if (candidate / "workshop").is_dir():
            return candidate
    raise RuntimeError(
        "Run from the repository root, notebooks/, or this module "
        "directory; or set WORKSHOP_NOTEBOOKS_DIR."
    )


NOTEBOOKS_ROOT = locate_notebooks_root()
REPO_ROOT = NOTEBOOKS_ROOT.parent
MODULE_DIR = NOTEBOOKS_ROOT / "06-neo4j-memory"
for path in (NOTEBOOKS_ROOT, MODULE_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
print(f"Workshop root: {REPO_ROOT}")


In [ ]:
import uuid
from contextlib import asynccontextmanager

import boto3

from memory_helpers import (
    DEMO_ID_PREFIX,
    HERO_HOTEL_NAME,
    WORKSHOP_OWNER,
    build_memory_client,
    get_actor_preferences_for_hotel,
    link_preference_to_message_and_hotel,
    load_config,
    tag_demo_records,
)

RUN_ID = uuid.uuid4().hex[:8]
ACTOR_A = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-alice"
ACTOR_B = f"{DEMO_ID_PREFIX}{RUN_ID}-guest-blake"
SESSION_A1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a1"
SESSION_A2 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-a2"
SESSION_B1 = f"{DEMO_ID_PREFIX}{RUN_ID}-session-b1"

try:
    config = load_config()
except RuntimeError as exc:
    config = None
    print(f"Neo4j is not configured: {exc}")

MEMORY_READY = (
    config is not None and boto3.Session().get_credentials() is not None
)

@asynccontextmanager
async def open_memory():
    memory = build_memory_client(config)
    await memory.connect()
    try:
        yield memory
    finally:
        await memory.close()

if MEMORY_READY:
    print(f"Run {RUN_ID}: Neo4j at {config.uri}, Bedrock in {config.region}.")
else:
    print("Not configured. Every live cell below will skip.")

## 2. Persist the preference statement

The notebook stores fixed message content to produce the same preference scenario on every run. Each run still creates new identifiers and records. This keeps the exercise focused on Neo4j. The first query verifies that the graph contains the required `Hotel` and reports a clear error when it is missing.

**Run the notebook from top to bottom once per session.** To start another run, execute the `RUN_ID` cell again. Reusing the same run ID creates duplicate message records.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        hotels = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (h:Hotel {name: $hotel_name})
            RETURN h.name AS name
            """,
            {"hotel_name": HERO_HOTEL_NAME},
        )
        if len(hotels) != 1:
            raise RuntimeError(
                f"The hotel graph must contain exactly one Hotel named "
                f"{HERO_HOTEL_NAME!r}; found {len(hotels)}."
            )

        preference_source = await memory.short_term.add_message(
            SESSION_A1,
            "user",
            f"I loved staying at {HERO_HOTEL_NAME}. A room on a high "
            "floor away from the elevator is a must for me.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_A1,
            "assistant",
            "I will remember that hotel and room preference.",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )

    print(f"Stored two fixture messages in {SESSION_A1}.")

## 3. Write one explicit preference and its provenance

The library creates a `Preference` owned by the actor. The workshop adds two relationships for inspection: `DERIVED_FROM` connects the preference to its source message, and `ABOUT_HOTEL` connects it to the existing `Hotel`. The workshop preserves the `Hotel` node without adding an `Entity` label or memory properties.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        preference = await memory.long_term.add_preference(
            category=f"hotels-demo08-{RUN_ID}",
            preference=(
                f"Loves {HERO_HOTEL_NAME} and wants a room on a high "
                "floor away from the elevator."
            ),
            context=f"Workshop run {RUN_ID}, session {SESSION_A1}",
            user_identifier=ACTOR_A,
        )

    linked = link_preference_to_message_and_hotel(
        config,
        str(preference.id),
        str(preference_source.id),
        HERO_HOTEL_NAME,
    )
    assert linked, "preference, message, or unique hero Hotel was missing"
    print("Preference linked to its source message and the real Hotel.")

## 4. Recall in a fresh session and isolate a second actor

Actor A starts `SESSION_A2` and asks a new question to demonstrate recall across sessions. The recall query starts at Actor A's `User` node and traverses the graph. Actor B starts a separate session, and the same actor-scoped traversal returns no preference. In production, the application must bind actor and session IDs to authenticated callers.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        await memory.short_term.add_message(
            SESSION_A2,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_A,
            extraction_mode="skip",
        )
        await memory.short_term.add_message(
            SESSION_B1,
            "user",
            "What hotel and room preference do you have for me?",
            user_identifier=ACTOR_B,
            extraction_mode="skip",
        )

    for_a = get_actor_preferences_for_hotel(
        config, ACTOR_A, HERO_HOTEL_NAME
    )
    for_b = get_actor_preferences_for_hotel(
        config, ACTOR_B, HERO_HOTEL_NAME
    )
    assert len(for_a) == 1, f"actor A expected one preference, got {len(for_a)}"
    assert not for_b, f"actor B unexpectedly saw {len(for_b)} preference(s)"
    print(f"Actor A in fresh session: {for_a[0]['preference']}")
    print("Actor B: no preference returned.")

## 5. Inspect the complete provenance path

Run one parameterized Cypher query to return the actor, preference, source message, source session, and canonical `Hotel`.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    async with open_memory() as memory:
        rows = await memory.query.cypher(
            """
            CYPHER 25
            MATCH (u:User {identifier: $actor})
                  -[:HAS_PREFERENCE]->(p:Preference)
                  -[:DERIVED_FROM]->(m:Message)
                  <-[:HAS_MESSAGE]-(c:Conversation),
                  (p)-[:ABOUT_HOTEL]->(h:Hotel {name: $hotel_name})
            RETURN u.identifier AS actor,
                   p.preference AS preference,
                   m.content AS source_message,
                   c.session_id AS source_session,
                   h.name AS hotel
            """,
            {"actor": ACTOR_A, "hotel_name": HERO_HOTEL_NAME},
        )
    assert len(rows) == 1, f"expected one provenance path, got {len(rows)}"
    for key, value in rows[0].items():
        print(f"{key:15s} {value}")

## Mark this run for scoped cleanup

The IDs use the `demo08-` namespace, and an ownership marker provides a second way to identify workshop data. Cleanup removes namespaced memory records, tagged orphaned preferences, and workshop-owned relationships. It preserves every `Hotel` node.

In [ ]:
if not MEMORY_READY:
    print("Skipping: no Neo4j or AWS configuration.")
else:
    marked = tag_demo_records(
        config,
        session_ids=[SESSION_A1, SESSION_A2, SESSION_B1],
        user_identifiers=[ACTOR_A, ACTOR_B],
    )
    print(f"Marked {marked} record(s) with {WORKSHOP_OWNER!r}.")

## Choose a Memory Architecture

| Dimension | AgentCore Memory in Module 4.2 | Neo4j graph memory in Module 6 |
|-----------|-----------------------------|----------------------------------|
| How memory is written | Managed extraction | Explicit application writes |
| When it is recallable | After asynchronous extraction | Immediately after the write |
| Inspectability | Retrieved through a service API | Queryable graph with source provenance |
| Domain linking | Separate from domain data | Workshop-owned edge to the real `Hotel` |
| Isolation | Actor namespaces managed by the service | Scoped writes and actor-anchored reads; application authorizes sessions |
| Operations | AWS operates the store | You operate Neo4j and the embedding contract |

Choose AgentCore Memory for managed extraction and operations. Choose graph memory when explicit writes, immediate visibility, provenance, and domain relationships matter.

## Optional notebook cleanup

The final cell can remove every run in the `demo08-` namespace while preserving `Hotel` nodes. Cleanup is disabled by default so **Run All** leaves the completed exercise available for inspection. Set `CLEAN_UP_DEMO_MEMORY` to `True` only when you are finished.

In [ ]:
from cleanup_memory import run_cleanup

CLEAN_UP_DEMO_MEMORY = False

if config is None:
    print("Cleanup skipped: Neo4j is not configured.")
elif not CLEAN_UP_DEMO_MEMORY:
    print("Cleanup skipped. Set CLEAN_UP_DEMO_MEMORY = True when finished.")
else:
    cleanup_result = run_cleanup(config)
    assert cleanup_result == 0, f"cleanup returned {cleanup_result}"
